In [ ]:
import requests
import pandas as pd 
import os

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for doc in documents_raw:
    course_name = doc['course']
    for doc in doc['documents']:
        doc['course'] = course_name
        documents.append(doc)


## RAG with min search

In [ ]:
import minsearch

index = minsearch.Index(
    text_fields=['question','text','section'],
    keyword_fields=['course'],
)
index.fit(documents)

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv("OPENROUTERAI_API_KEY")
)

In [ ]:
def search(query):
    boost = {'question': 3.0, 'section': 1.0}

    results = index.search(query, filter_dict={'course':'data-engineering-zoomcamp'}, boost_dict=boost, num_results=5)
    return results

def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context_template = """
Q: {question}
A: {text}
""".strip()
    
    context = ""
    for result in search_results:
        context += context_template.format(question=result['question'], text=result['text']) + "\n"
    
    prompt = prompt_template.format(question=query, context=context)
    return prompt

def llm(prompt):
    response = client.chat.completions.create(
      model="qwen/qwen3-4b:free",
      messages=[
        {
          "role": "user",
          "content": prompt,
        },
      ]
    )

    return response.choices[0].message.content

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

query = 'how do I enroll the course?'
answer = rag(query)



## RAG with vector search
Pull the image and start the container using the following commands:
```bash
docker pull qdrant/qdrant

docker run -p 6333:6333 -p 6334:6334 \
   -v "$(pwd)/qdrant_storage:/qdrant/storage:z" \
   qdrant/qdrant
```

In [ ]:
from qdrant_client import QdrantClient, models

In [ ]:
qd_client = QdrantClient("http://localhost:6333")

In [ ]:
# Initial hyperparameters
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [ ]:
collection_name = "zoomcamp-rag"

qd_client.delete_collection(collection_name=collection_name)

qd_client.create_collection(
    collection_name = collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

In [ ]:
points = []

for i,doc in enumerate(documents):
    text = doc['text'] + ' ' + doc['question']
    point = models.PointStruct(
        id=i,
        vector=models.Document(text=text, model=model_handle),
        payload=doc
    )
    points.append(point)

In [ ]:
qd_client.upsert(
    collection_name=collection_name,
    points=points
)

In [ ]:
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword" # exact matching on string metadata fields
)

In [ ]:
question = 'I just discovered the course. Can I still join it?'

In [ ]:
def vector_search(query, course='data-engineering-zoomcamp'):
    print('vector search is used')

    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_handle
        ),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=5,
        with_payload=True
    )

    results=[]
    for point in query_points.points:
        results.append(point.payload)

    return results


In [ ]:
def rag(query):
    search_results = vector_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [ ]:
rag('how do I run kafka?')